
# RAG Colab: FAISS + BM25 + Qwen2.5-3B + LoRA Adapter



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

# 1) Install dependencies
!pip -q install -U sentence-transformers faiss-cpu rank-bm25 langchain-community langchain-core pandas openpyxl scikit-learn tqdm transformers accelerate peft bitsandbytes opensearch-py


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [ ]:
# Run this cell to uninstall torchao and kill the process, which will allow you to restart the runtime and use the newly installed dependencies.
# After run this cell, please run the next cell to continue with the rest of the code.
# Just run the next cell, dont run the above cell again
!pip uninstall -y torchao
import os
os.kill(os.getpid(), 9)

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:

# 2) Config
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/LLM_RAG_PROJECT_BASE_QLoRA")

FAISS_INDEX_DIR = ROOT / "data/faiss/law_chunks_e5_base"
CHUNKS_JSONL = ROOT / "data/datahuggingface/corpus_important_docs_chunks.jsonl"
EVAL_CSV = ROOT / "data/evaluate/evaluate.csv"

RUN_NAME = "eval150_adapter_qwen25_3b"
OUTPUT_DIR = ROOT / "analysis/eval_runs"
EVAL_LIMIT = 100  # <=150 theo requirement

# retrieval
BM25_K = 30
FAISS_K = 30
RRF_K = 60
FINAL_K = 5
MAX_CHUNKS_PER_DOC = 2
MIN_QUERY_TERM_OVERLAP = 2
KEEP_AT_LEAST = 5
CONTEXT_NEIGHBOR_WINDOW = 1
CONTEXT_SAME_ARTICLE_LIMIT = 2
CONTEXT_MAX_ADDITIONAL = 4
MAX_CONTEXT_TOKENS = 2200

DOC_FOCUS_MODE = "auto"  # off / auto / strict
DOC_FOCUS_MAX_PRIMARY_DOCS = 1
DOC_FOCUS_PRIMARY_MARGIN = 0.12
DOC_FOCUS_PRIMARY_DOC_QUOTA = 4
DOC_FOCUS_OTHER_DOC_QUOTA = 1

# embeddings + reranker
E5_MODEL = "intfloat/multilingual-e5-base"
METRIC_EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
DEVICE = "cuda"
PREFIX_MODE = "query_passage"
NORMALIZE = True

USE_RERANKER = True
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
RERANKER_DEVICE = "cuda"  # set "cuda" nếu đủ VRAM
RERANKER_TOP_N = 20

# LLM local with adapter
BASE_LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "huyle1901/qwen25-3b-legal-qlora"  # đổi theo repo adapter của bạn
LLM_LOCAL_DEVICE = "cuda"
LLM_LOCAL_DTYPE = "auto"  # auto/float16/bfloat16/float32
LLM_TRUST_REMOTE_CODE = False

TEMPERATURE = 0.0
MAX_TOKENS = 768


for p in [FAISS_INDEX_DIR, CHUNKS_JSONL, EVAL_CSV]:
    print(p, "OK" if p.exists() else "MISSING")


/content/drive/MyDrive/LLM_RAG_PROJECT_FINAL/data/faiss/law_chunks_e5_base OK
/content/drive/MyDrive/LLM_RAG_PROJECT_FINAL/data/datahuggingface/corpus_important_docs_chunks.jsonl OK
/content/drive/MyDrive/LLM_RAG_PROJECT_FINAL/data/evaluate/evaluate.csv OK


In [2]:

# 3) Imports from current pipeline
import json
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import FAISS
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from rag_hybrid_opensearch_faiss_qwen import (
    SYSTEM_PROMPT,
    apply_document_focus,
    E5Embeddings,
    RetrievedItem,
    build_answer_prompt,
    build_context,
    expand_context_chunks,
    enrich_and_filter_by_relevance,
    extract_target_article_numbers,
    extract_target_doc_numbers,
    extract_query_terms,
    limit_chunks_per_doc,
    load_chunk_store,
    rerank_chunks,
    rrf_fuse,
)

from evaluate_rag_random_subset import (
    fix_mojibake,
    fold_text,
    normalize_text,
    jaccard_similarity,
    token_overlap_score,
    bleu_score_simple,
    rouge_l_score,
)


In [3]:
import rag_hybrid_opensearch_faiss_qwen as rag

rag.SYSTEM_PROMPT = (
    "Bạn là trợ lý pháp lý Việt Nam. "
    "Chỉ được sử dụng thông tin trong CONTEXT, không dùng kiến thức bên ngoài. "
    "Mục tiêu là TRÍCH XUẤT ĐẦY ĐỦ quy định liên quan, không trả lời tóm tắt thiếu ý. "
    "Nếu câu hỏi nêu rõ số hiệu văn bản (ví dụ 02/2025/TT-NHNN), phải ưu tiên văn bản đó làm căn cứ chính. "
    "Chỉ dùng văn bản khác khi CONTEXT thể hiện rõ là văn bản liên quan trực tiếp (sửa đổi, dẫn chiếu, điều kiện áp dụng). "
    "Nếu điều khoản có danh sách điểm/khoản (a, b, c... hoặc 1, 2, 3...), phải liệt kê đủ các ý liên quan đến câu hỏi; không được bỏ sót. "
    "Không tự thêm nội dung không có trong CONTEXT. "
    "Không dùng tiếng Anh trong tiêu đề/nội dung trả lời. "
    "Nếu dữ liệu thiếu, nêu rõ phần thiếu và viết đúng câu: 'Không đủ dữ liệu trong CONTEXT'. "
    "Luôn gắn trích dẫn nguồn dạng [1], [2], ... cho từng ý quan trọng."
)

rag.ANSWER_REQUIREMENTS = (
    "Yêu cầu trả lời:\n"
    "1) Mở đầu bằng câu 'Căn cứ ...' và nêu đúng văn bản + điều/khoản/điểm chính.\n"
    "2) Trả lời đúng trọng tâm câu hỏi, theo hướng trích xuất quy định:\n"
    "   - Nếu hỏi 'trường hợp nào/điều kiện nào/nghĩa vụ gì' => liệt kê đầy đủ từng trường hợp/điều kiện/nghĩa vụ.\n"
    "   - Nếu hỏi kèm 'quy định/thủ tục như thế nào' => nêu thêm phần thủ tục, trình tự, lưu ý áp dụng (nếu có trong CONTEXT).\n"
    "3) Giữ sát câu chữ pháp lý trong CONTEXT; không diễn giải suy đoán.\n"
    "4) Nếu có ngoại lệ/giới hạn/phạm vi áp dụng trong điều khoản thì phải nêu đủ.\n"
    "5) Kết thúc bằng 'Tóm lại:' với kết luận ngắn gọn, đúng với các ý đã liệt kê ở trên.\n"
    "6) Không được kết thúc khi chưa liệt kê đủ các ý liên quan trong điều/khoản chính.\n"
    "7) Không dùng cách ghi sai định dạng như 'Điều 12.0' hoặc 'Khoản 1.0'; ghi đúng 'Điều 12', 'khoản 1'.\n"
    "8) Mỗi ý quan trọng phải có trích dẫn [n]. Nếu không đủ dữ liệu thì ghi: 'Không đủ dữ liệu trong CONTEXT'.\n"
)

In [4]:

# 4) Build local BM25 index + load FAISS
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]{2,}", fold_text(text))

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

chunk_rows = read_jsonl(CHUNKS_JSONL)
bm25_corpus_tokens = [tokenize(str(r.get("chunk_text") or "")) for r in chunk_rows]
bm25 = BM25Okapi(bm25_corpus_tokens)

emb = E5Embeddings(
    model_name=E5_MODEL,
    device=DEVICE,
    normalize=NORMALIZE,
    prefix_mode=PREFIX_MODE,
)
metric_emb = E5Embeddings(
    model_name=METRIC_EMBEDDING_MODEL,
    device=DEVICE,
    normalize=True,
    prefix_mode=PREFIX_MODE,
)
faiss_vs = FAISS.load_local(str(FAISS_INDEX_DIR), emb, allow_dangerous_deserialization=True)
chunk_store = load_chunk_store(str(CHUNKS_JSONL.resolve()))

print("chunks:", len(chunk_rows))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


chunks: 34374


In [5]:

# 5) Local LLM with base + adapter
_LOCAL_ADAPTER_PIPELINE_CACHE: dict[tuple, tuple] = {}

def get_local_adapter_pipeline(
    base_model: str,
    adapter_repo: str,
    device: str,
    dtype: str,
    trust_remote_code: bool,
):
    key = (base_model, adapter_repo, device, dtype, trust_remote_code)
    cached = _LOCAL_ADAPTER_PIPELINE_CACHE.get(key)
    if cached is not None:
        return cached

    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

    run_device = device
    if run_device.startswith("cuda") and not torch.cuda.is_available():
        print("CUDA is not available for local adapter LLM. Falling back to CPU.")
        run_device = "cpu"

    dtype_map = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }

    model_kwargs: dict[str, Any] = {"trust_remote_code": trust_remote_code}
    if dtype in dtype_map:
        model_kwargs["torch_dtype"] = dtype_map[dtype]
    elif dtype == "auto" and run_device.startswith("cuda"):
        model_kwargs["torch_dtype"] = torch.float16

    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=trust_remote_code)
    base = AutoModelForCausalLM.from_pretrained(base_model, **model_kwargs)
    model = PeftModel.from_pretrained(base, adapter_repo)

    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    device_index = 0 if run_device.startswith("cuda") else -1
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=device_index,
    )
    _LOCAL_ADAPTER_PIPELINE_CACHE[key] = (generator, tokenizer)
    return generator, tokenizer

def generate_with_adapter(question: str, context: str) -> str:
    prompt = build_answer_prompt(question, context)
    generator, tokenizer = get_local_adapter_pipeline(
        base_model=BASE_LLM_MODEL,
        adapter_repo=ADAPTER_REPO,
        device=LLM_LOCAL_DEVICE,
        dtype=LLM_LOCAL_DTYPE,
        trust_remote_code=LLM_TRUST_REMOTE_CODE,
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt_text = f"{SYSTEM_PROMPT}\n\n{prompt}\n\nTra loi:"

    do_sample = TEMPERATURE > 0
    gen_kwargs: dict[str, Any] = {
        "max_new_tokens": MAX_TOKENS,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if tokenizer.eos_token_id is not None:
        gen_kwargs["eos_token_id"] = tokenizer.eos_token_id
    if do_sample:
        gen_kwargs["temperature"] = max(TEMPERATURE, 1e-5)

    out = generator(prompt_text, **gen_kwargs)
    text = out[0]["generated_text"]
    if text.startswith(prompt_text):
        text = text[len(prompt_text):]
    return text.strip()


In [6]:

# 6) Evaluate fixed evaluate.csv (<=150 rows)
def search_bm25_local(query: str, top_k: int) -> list[RetrievedItem]:
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    if len(scores) == 0:
        return []

    idxs = np.argsort(scores)[::-1][:top_k]
    out: list[RetrievedItem] = []
    rank = 1
    for i in idxs:
        s = float(scores[i])
        if s <= 0:
            continue
        row = chunk_rows[int(i)]
        cid = str(row.get("chunk_id") or "")
        if not cid:
            continue
        out.append(
            RetrievedItem(
                chunk_id=cid,
                payload=row,
                source="bm25_local",
                rank=rank,
                raw_score=s,
            )
        )
        rank += 1
    return out

def search_faiss_loaded(query: str, top_k: int) -> list[RetrievedItem]:
    hits = faiss_vs.similarity_search_with_score(query, k=top_k)
    out: list[RetrievedItem] = []
    for rank, (doc, score) in enumerate(hits, start=1):
        meta = doc.metadata or {}
        payload = dict(meta)
        payload.setdefault("chunk_text", doc.page_content)
        cid = str(payload.get("chunk_id") or "")
        if not cid:
            continue
        out.append(
            RetrievedItem(
                chunk_id=cid,
                payload=payload,
                source="faiss",
                rank=rank,
                raw_score=float(score),
            )
        )
    return out

def rag_answer_for_question(question: str) -> tuple[str, list[dict[str, Any]]]:
    bm25_hits = search_bm25_local(question, BM25_K)
    faiss_hits = search_faiss_loaded(question, FAISS_K)

    target_docs = extract_target_doc_numbers(question)
    target_articles = extract_target_article_numbers(question)

    fused = rrf_fuse([bm25_hits, faiss_hits], k=RRF_K)
    query_terms = extract_query_terms(question)
    fused = enrich_and_filter_by_relevance(
        fused,
        query_terms=query_terms,
        min_overlap=MIN_QUERY_TERM_OVERLAP,
        keep_at_least=KEEP_AT_LEAST,
        required_doc_numbers=target_docs,
        required_article_numbers=target_articles,
    )
    fused = rerank_chunks(
        query=question,
        chunks=fused,
        use_reranker=USE_RERANKER,
        reranker_model=RERANKER_MODEL,
        reranker_device=RERANKER_DEVICE,
        reranker_top_n=RERANKER_TOP_N,
    )
    fused = apply_document_focus(
        chunks=fused,
        mode=DOC_FOCUS_MODE,
        required_doc_numbers=target_docs,
        max_primary_docs=DOC_FOCUS_MAX_PRIMARY_DOCS,
        primary_score_margin=DOC_FOCUS_PRIMARY_MARGIN,
        primary_doc_quota=DOC_FOCUS_PRIMARY_DOC_QUOTA,
        other_doc_quota=DOC_FOCUS_OTHER_DOC_QUOTA,
    )
    fused = limit_chunks_per_doc(fused, max_chunks_per_doc=MAX_CHUNKS_PER_DOC)
    base_chunks = fused[:FINAL_K]
    if not base_chunks:
        return "", []

    final_chunks = expand_context_chunks(
        base_chunks=base_chunks,
        store=chunk_store,
        neighbor_window=CONTEXT_NEIGHBOR_WINDOW,
        same_article_limit=CONTEXT_SAME_ARTICLE_LIMIT,
        max_additional=CONTEXT_MAX_ADDITIONAL,
    )
    context = build_context(final_chunks, max_context_tokens=MAX_CONTEXT_TOKENS)
    answer = generate_with_adapter(question, context)
    return answer, final_chunks

def cosine_similarity_score(pred: str, true: str) -> float:
    if not pred or not true:
        return 0.0
    pv = metric_emb.embed_query(pred)
    tv = metric_emb.embed_query(true)
    return float(cosine_similarity([pv], [tv])[0][0])

eval_df = pd.read_csv(EVAL_CSV, encoding="utf-8-sig")
if EVAL_LIMIT > 0:
    eval_df = eval_df.head(EVAL_LIMIT).copy()

required_cols = {"row_index", "question", "ground_truth", "vbpl_extracted"}
missing_cols = required_cols.difference(eval_df.columns)
if missing_cols:
    raise ValueError(f"Missing columns in evaluate.csv: {missing_cols}")

results: list[dict[str, Any]] = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
    question = fix_mojibake(str(row["question"])).strip()
    ground_truth = fix_mojibake(str(row["ground_truth"])).strip()
    vbpl = fix_mojibake(str(row.get("vbpl_extracted", ""))).strip()

    try:
        pred, final_chunks = rag_answer_for_question(question)
        err = ""
    except Exception as e:
        pred = ""
        final_chunks = []
        err = str(e)

    metrics = {
        "Cosine_Similarity": cosine_similarity_score(pred, ground_truth),
        "Jaccard_Similarity": jaccard_similarity(pred, ground_truth),
        "Token_Overlap": token_overlap_score(pred, ground_truth),
        "BLEU_Score": bleu_score_simple(pred, ground_truth),
        "ROUGE_L": rouge_l_score(pred, ground_truth),
    }

    results.append(
        {
            "row_index": int(row["row_index"]),
            "question": question,
            "ground_truth": ground_truth,
            "vbpl_extracted": vbpl,
            "prediction": pred,
            "retrieved_docs": "|".join(
                str((ch.get("payload", {}) or {}).get("document_number") or "")
                for ch in final_chunks
            ),
            "error": err,
            **metrics,
        }
    )

results_df = pd.DataFrame(results)
mean_metrics = {
    k: float(results_df[k].mean())
    for k in ["Cosine_Similarity", "Jaccard_Similarity", "Token_Overlap", "BLEU_Score", "ROUGE_L"]
}

out_dir = OUTPUT_DIR / RUN_NAME
out_dir.mkdir(parents=True, exist_ok=True)
results_csv = out_dir / "rag_eval_results.csv"
summary_json = out_dir / "summary.json"

results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")

summary = {
    "source_eval_csv": str(EVAL_CSV),
    "eval_rows": int(len(eval_df)),
    "mean_metrics": mean_metrics,
    "config": {
        "bm25_backend": "local_rank_bm25",
        "faiss_index_dir": str(FAISS_INDEX_DIR),
        "chunks_jsonl": str(CHUNKS_JSONL),
        "llm_base_model": BASE_LLM_MODEL,
        "llm_adapter_repo": ADAPTER_REPO,
        "reranker_device": RERANKER_DEVICE,
    },
    "outputs": {
        "results_csv": str(results_csv),
        "summary_json": str(summary_json),
    },
}
summary_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("=== FIXED EVAL SUMMARY ===")
print(f"rows={len(eval_df)} from {EVAL_CSV}")
for k, v in mean_metrics.items():
    print(f"{k}: {v:.4f}")
print(f"saved: {results_csv}")


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluating:  10%|█         | 10/100 [17:33<1:57:52, 78.58s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the d

Evaluating:  73%|███████▎  | 73/100 [1:41:34<39:41, 88.19s/it]Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating:  86%|████████▌ | 86/100 [1:58:49<20:07, 86.26s/it]Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating: 100%|██████████| 100/100 [2:16:19<00:00, 81.79s/it]


=== FIXED EVAL SUMMARY ===
rows=100 from /content/drive/MyDrive/LLM_RAG_PROJECT_FINAL/data/evaluate/evaluate.csv
Cosine_Similarity: 0.9603
Jaccard_Similarity: 0.5422
Token_Overlap: 0.7274
BLEU_Score: 0.4575
ROUGE_L: 0.4499
saved: /content/drive/MyDrive/LLM_RAG_PROJECT_FINAL/analysis/eval_runs/eval150_adapter_qwen25_3b/rag_eval_results.csv
